Before running the notebook run the command `mlflow server --host 127.0.0.1 --port 5000`

In [13]:
import json
from pathlib import Path

import mlflow
import pandas as pd
from hydra import compose, initialize
from pycaret import classification


# Load config

In [14]:
with initialize(version_base=None, config_path="../conf"):
    config = compose(config_name="health_prediction_config")

model_folder_path = Path(Path("..") / config.model_path)
model_folder_path.mkdir(parents=True, exist_ok=True)

# Load data

In [15]:
health_df = pd.read_csv(Path("..") / config.silver_dataset_path)
health_df

,Age,Gender,Cholesterol,Glucose,Smoking,Alcohol Consumption,Exercise,BMI,Family History,Heart Disease,...,Kidney Disease,Cancer,Alzheimer's Disease,COPD,Liver Disease,Parkinson's Disease,Tuberculosis,Blood Pressure_Low,Blood Pressure_Normal,Disease
0,69,0,1,1,1,0,0,35.671099,0,1,...,0,1,0,0,0,0,0,0,0,1
1,32,0,1,0,1,0,1,38.554188,1,0,...,0,0,0,0,1,0,0,1,0,1
2,89,1,1,0,0,0,1,18.932964,1,1,...,0,0,0,0,0,0,0,0,1,1
3,78,0,1,1,0,0,1,21.806350,1,0,...,0,1,0,0,1,0,0,0,0,1
4,38,0,0,0,1,1,1,37.552683,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,27,1,1,0,0,0,0,31.960176,1,1,...,0,0,0,0,0,0,0,1,0,1
996,51,1,1,0,0,1,1,20.118492,1,0,...,0,0,0,0,0,0,0,0,0,0
997,72,1,1,0,1,0,0,20.916536,1,0,...,0,0,0,0,0,0,0,0,1,0
998,49,0,1,1,1,0,1,19.560143,1,0,...,0,0,1,0,0,0,0,0,1,1


# Modelling

In [16]:
a = [1, 2, 3, 4, 5, 6, 7, 8, 9]
b = [9, 8, 7, 6, 5, 4, 3, 2, 1]
c = [9, 1, 8, 2, 7, 3, 6, 4, 5]

list(zip(a, b, c))

[(1, 9, 9),
 (2, 8, 1),
 (3, 7, 8),
 (4, 6, 2),
 (5, 5, 7),
 (6, 4, 3),
 (7, 3, 6),
 (8, 2, 4),
 (9, 1, 5)]

In [17]:
# Connect to Mlflow server
mlflow.set_tracking_uri("http://127.0.0.1:5000")


# Initialize PyCaret Setup and train models
target_cols = ["Disease", "Heart Disease", "Diabetes", "Stroke", "Kidney Disease", "Cancer", "Alzheimer's Disease", "COPD", "Liver Disease", "Parkinson's Disease", "Tuberculosis"]

reg_setups = {"with": {}, "without": {}}
models = {"with": {}, "without": {}, "top_10": {}}
comp_results = {"with": {}, "without": {}}

for col in target_cols:
    experiment_name = "health_" + col.lower().replace(" ", "_").replace("'", " ") + "_with"
    other_target_cols = [tcol for tcol in target_cols if tcol != col]

    # Setup with "Age" and "BMI" columns
    reg_setups["with"][col] = classification.setup(
        data=health_df,
        target=col,
        train_size=config.train_size,
        ignore_features=other_target_cols,
        fix_imbalance=True,
        transformation=True,
        normalize=True,
        session_id=config.random_state,
        log_experiment=True,
        experiment_name=experiment_name,
        memory=False,
    )
    # Train and look at top 10 models
    models["with"][col] = list(
        zip(
            classification.compare_models(sort="AUC", n_select=10, exclude=["dummy"]),
            classification.pull()["AUC"],
            ["with"] * 10,
        )
    )

    # Setup without "Age" and "BMI" columns
    reg_setups["without"][col] = classification.setup(
        data=health_df,
        target=col,
        train_size=config.train_size,
        ignore_features=["Age", "BMI"] + other_target_cols,
        fix_imbalance=True,
        transformation=True,
        normalize=True,
        session_id=config.random_state,
        log_experiment=True,
        experiment_name=experiment_name + "out",
        memory=False,
    )
    # Train and look at top 10 models
    models["without"][col] = list(zip(classification.compare_models(sort="AUC", n_select=10, exclude=["dummy"]), classification.pull()["AUC"], ["without"] * 10))

    # Combine both into top 10
    models["top_10"][col] = sorted(models["without"][col] + models["with"][col], key=lambda e: e[1], reverse=True)[:10]


2026/08/23 10:06:15 INFO mlflow.tracking._tracking_service.client: 🏃 View run Session Initialized 515b at: http://127.0.0.1:5000/#/experiments/494945949872346356/runs/6d61927fb7a6483584645de09bb46609.
2026/08/23 10:06:15 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/494945949872346356.
2026/08/23 10:06:15 INFO mlflow.tracking._tracking_service.client: 🏃 View run Session Initialized 641b at: http://127.0.0.1:5000/#/experiments/296219890235609537/runs/03367888a0d6436aac0cbed1b670464e.
2026/08/23 10:06:15 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/296219890235609537.
2026/08/23 10:06:16 INFO mlflow.tracking._tracking_service.client: 🏃 View run Session Initialized fade at: http://127.0.0.1:5000/#/experiments/148140050682696434/runs/ca9953f3dbf8474c89cf49fcdd2ede07.
2026/08/23 10:06:16 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5

,Description,Value
0,Session id,67
1,Target,Disease
2,Target type,Binary
3,Original data shape,"(1000, 22)"
4,Transformed data shape,"(1410, 12)"
5,Transformed train set shape,"(1210, 12)"
6,Transformed test set shape,"(200, 12)"
7,Ignore features,10
8,Numeric features,11
9,Preprocess,True


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.7412,0.5804,0.9503,0.7648,0.8473,0.0555,0.0683,0.0970
xgboost,Extreme Gradient Boosting,0.6963,0.5480,0.8547,0.7695,0.8094,0.0682,0.0727,0.0740
qda,Quadratic Discriminant Analysis,0.5937,0.5347,0.6612,0.7699,0.7103,0.0404,0.0427,0.0300
catboost,CatBoost Classifier,0.7263,0.5301,0.9290,0.7620,0.8369,0.0298,0.0266,2.6090
nb,Naive Bayes,0.6062,0.5133,0.6896,0.7663,0.7241,0.0384,0.0405,0.7670
lr,Logistic Regression,0.5000,0.5066,0.5292,0.7340,0.6134,-0.0447,-0.0524,0.8540
ridge,Ridge Classifier,0.5000,0.5064,0.5292,0.7340,0.6134,-0.0447,-0.0524,0.0350
lda,Linear Discriminant Analysis,0.5000,0.5063,0.5292,0.7340,0.6134,-0.0447,-0.0524,0.0320
rf,Random Forest Classifier,0.7262,0.5052,0.9340,0.7595,0.8375,0.0217,0.0305,0.1170
et,Extra Trees Classifier,0.6912,0.5021,0.8795,0.7536,0.8111,-0.0152,-0.0202,0.0980


2026/08/23 10:07:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:07:35 INFO mlflow.tracking._tracking_service.client: 🏃 View run Gradient Boosting Classifier at: http://127.0.0.1:5000/#/experiments/494945949872346356/runs/a837ea2b07d84818a790784fcbe9fd22.
2026/08/23 10:07:35 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/494945949872346356.
2026/08/23 10:07:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:07:39 INFO mlflow.tracking._tracking_service.client: 🏃 View run Extreme Gradient Boosting at: http://127.0.0.1:5000/#/experiments/494945949872346356/runs/3c511de174814348aa07d257daa36ac0.
2026/08/23 10:07:39 INFO mlflow.tracking._tracking_servi

,Description,Value
0,Session id,67
1,Target,Disease
2,Target type,Binary
3,Original data shape,"(1000, 22)"
4,Transformed data shape,"(1410, 10)"
5,Transformed train set shape,"(1210, 10)"
6,Transformed test set shape,"(200, 10)"
7,Ignore features,12
8,Numeric features,9
9,Preprocess,True


...

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.9637,0.5717,0.0000,0.0000,0.0000,-0.0019,-0.0022,0.0730
nb,Naive Bayes,0.7850,0.5686,0.3167,0.0548,0.0930,0.0364,0.0548,0.0200
knn,K Neighbors Classifier,0.7400,0.5637,0.2833,0.0356,0.0629,0.0020,0.0100,0.0340
rf,Random Forest Classifier,0.9637,0.5632,0.0000,0.0000,0.0000,-0.0019,-0.0022,0.0640
qda,Quadratic Discriminant Analysis,0.8800,0.5417,0.0000,0.0000,0.0000,-0.0488,-0.0562,0.0190
catboost,CatBoost Classifier,0.9600,0.5392,0.0000,0.0000,0.0000,-0.0069,-0.0076,1.6770
ada,Ada Boost Classifier,0.9588,0.5333,0.0000,0.0000,0.0000,-0.0088,-0.0098,0.0420
xgboost,Extreme Gradient Boosting,0.9588,0.5163,0.0000,0.0000,0.0000,-0.0077,-0.0083,0.0370
svm,SVM - Linear Kernel,0.5363,0.4830,0.4167,0.0294,0.0548,-0.0110,-0.0205,0.0220
dt,Decision Tree Classifier,0.9150,0.4741,0.0000,0.0000,0.0000,-0.0407,-0.0426,0.0220


2026/08/23 10:27:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:27:48 INFO mlflow.tracking._tracking_service.client: 🏃 View run Gradient Boosting Classifier at: http://127.0.0.1:5000/#/experiments/668694462194489806/runs/7574cd6e38134173bd418f1af501429d.
2026/08/23 10:27:48 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/668694462194489806.
2026/08/23 10:27:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:27:50 INFO mlflow.tracking._tracking_service.client: 🏃 View run Naive Bayes at: http://127.0.0.1:5000/#/experiments/668694462194489806/runs/f372b1eb31e9437583be35dc17fc2271.
2026/08/23 10:27:50 INFO mlflow.tracking._tracking_service.client: 🧪 V

,Description,Value
0,Session id,67
1,Target,Tuberculosis
2,Target type,Binary
3,Original data shape,"(1000, 22)"
4,Transformed data shape,"(1744, 10)"
5,Transformed train set shape,"(1544, 10)"
6,Transformed test set shape,"(200, 10)"
7,Ignore features,12
8,Numeric features,9
9,Preprocess,True


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
qda,Quadratic Discriminant Analysis,0.8050,0.4475,0.1500,0.0298,0.0486,-0.0094,-0.0120,0.0170
dt,Decision Tree Classifier,0.9487,0.4437,0.0000,0.0000,0.0000,-0.0192,-0.0208,0.0170
gbc,Gradient Boosting Classifier,0.9625,0.4387,0.0000,0.0000,0.0000,-0.0036,-0.0040,0.0470
svm,SVM - Linear Kernel,0.5788,0.4372,0.3167,0.0204,0.0382,-0.0277,-0.0442,0.0190
et,Extra Trees Classifier,0.9512,0.4225,0.0000,0.0000,0.0000,-0.0174,-0.0192,0.0550
knn,K Neighbors Classifier,0.7250,0.4177,0.1667,0.0201,0.0356,-0.0274,-0.0420,0.0290
rf,Random Forest Classifier,0.9500,0.4030,0.0000,0.0000,0.0000,-0.0186,-0.0202,0.0660
nb,Naive Bayes,0.6175,0.4005,0.2667,0.0224,0.0412,-0.0241,-0.0437,0.0150
xgboost,Extreme Gradient Boosting,0.9525,0.3814,0.0000,0.0000,0.0000,-0.0145,-0.0156,0.0300
ada,Ada Boost Classifier,0.9650,0.3792,0.0000,0.0000,0.0000,0.0000,0.0000,0.0350


2026/08/23 10:28:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:28:43 INFO mlflow.tracking._tracking_service.client: 🏃 View run Quadratic Discriminant Analysis at: http://127.0.0.1:5000/#/experiments/479745575655938716/runs/c851c1d1428b4276a11db77e2117f653.
2026/08/23 10:28:43 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/479745575655938716.
2026/08/23 10:28:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:28:45 INFO mlflow.tracking._tracking_service.client: 🏃 View run Decision Tree Classifier at: http://127.0.0.1:5000/#/experiments/479745575655938716/runs/c8925d0eb8f0438fa85d1d5b7d8ca8d1.
2026/08/23 10:28:45 INFO mlflow.tracking._tracking_ser

In [18]:
for col, col_models in models["top_10"].items():
    print(col)
    for pos, _ in enumerate(col_models, start=1):
        model, untuned_auc, setup_name = _
        model_name = type(model).__name__
        print(f" {pos: >2}. {model_name: <30}: {untuned_auc:0<6}")

    print()

Disease
  1. GradientBoostingClassifier    : 0.5804
  2. XGBClassifier                 : 0.5480
  3. QuadraticDiscriminantAnalysis : 0.5347
  4. QuadraticDiscriminantAnalysis : 0.5307
  5. CatBoostClassifier            : 0.5301
  6. GradientBoostingClassifier    : 0.5211
  7. GaussianNB                    : 0.5133
  8. AdaBoostClassifier            : 0.5089
  9. LogisticRegression            : 0.5066
 10. RidgeClassifier               : 0.5064

Heart Disease
  1. GradientBoostingClassifier    : 0.5617
  2. QuadraticDiscriminantAnalysis : 0.5525
  3. RidgeClassifier               : 0.5502
  4. LinearDiscriminantAnalysis    : 0.5501
  5. LogisticRegression            : 0.5500
  6. AdaBoostClassifier            : 0.5499
  7. GaussianNB                    : 0.5469
  8. GradientBoostingClassifier    : 0.5464
  9. GaussianNB                    : 0.5450
 10. QuadraticDiscriminantAnalysis : 0.5409

Diabetes
  1. DecisionTreeClassifier        : 0.5707
  2. RandomForestClassifier        : 0.5703

## Tuning

In [19]:
best_models = {}

for col, col_models in models["top_10"].items():
    max_auc = -1
    best_model = None

    for pos, (untuned_model, untuned_auc, setup_name) in enumerate(col_models, start=1):
        current_setup = reg_setups[setup_name][col]

        model_name = type(untuned_model).__name__.lower()
        base_name = f"{col}_{setup_name}_{pos}_{model_name}"
        base_name = base_name.replace(" ", "-").replace("'", "").lower()

        # Tune model
        tuned_model = current_setup.tune_model(
            untuned_model,
            optimize="AUC",
            n_iter=25,
            choose_better=False,
        )
        tuned_auc = current_setup.pull().loc["Mean", "AUC"]

        # Choose the best tuned model if it outperforms the base model
        if tuned_auc >= untuned_auc:
            selected_model = tuned_model
            selected_auc = tuned_auc
        else:
            selected_model = untuned_model
            selected_auc = untuned_auc

        # Choose the best model in this column
        if selected_auc > max_auc:
            max_auc = selected_auc

            calibrated_model = current_setup.calibrate_model(selected_model, method="sigmoid")

            best_model = (calibrated_model, selected_auc, setup_name)

    best_models[col] = best_model

    chosen_setup = reg_setups[best_model[2]][col]
    chosen_setup.save_model(
        best_model[0],
        str(model_folder_path / col).replace(" ", "_").replace("'", "-"),
    )

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7000,0.6083,0.8197,0.7937,0.8065,0.1406,0.1409
1,0.7125,0.5988,0.8852,0.7714,0.8244,0.0515,0.0555
2,0.6000,0.3676,0.7705,0.7231,0.7460,-0.1907,-0.1928
3,0.7750,0.7153,0.8689,0.8413,0.8548,0.3554,0.3563
4,0.7875,0.6238,0.9344,0.8143,0.8702,0.2990,0.3220
5,0.7625,0.6275,0.9333,0.7887,0.8550,0.2245,0.2512
6,0.7000,0.6375,0.8500,0.7727,0.8095,0.1111,0.1140
7,0.7125,0.5342,0.9000,0.7606,0.8244,0.0612,0.0685
8,0.7250,0.5392,0.8833,0.7794,0.8281,0.1538,0.1617


Fitting 10 folds for each of 25 candidates, totalling 250 fits


2026/08/23 10:29:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:29:29 INFO mlflow.tracking._tracking_service.client: 🏃 View run Gradient Boosting Classifier at: http://127.0.0.1:5000/#/experiments/494945949872346356/runs/ca88e9c1b7254e8d944538b98375e425.
2026/08/23 10:29:29 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/494945949872346356.


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7875,0.6212,1.0000,0.7821,0.8777,0.1521,0.2869
1,0.7500,0.5047,0.9836,0.7595,0.8571,-0.0243,-0.0628
2,0.7500,0.4763,0.9836,0.7595,0.8571,-0.0243,-0.0628
3,0.7500,0.6756,0.9836,0.7595,0.8571,-0.0243,-0.0628
4,0.7625,0.5617,1.0000,0.7625,0.8652,0.0000,0.0000
5,0.7500,0.6817,1.0000,0.7500,0.8571,0.0000,0.0000
6,0.7625,0.6450,0.9833,0.7662,0.8613,0.1163,0.1899
7,0.7500,0.5208,1.0000,0.7500,0.8571,0.0000,0.0000
8,0.7375,0.5083,0.9833,0.7468,0.8489,-0.0244,-0.0650


2026/08/23 10:29:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:29:36 INFO mlflow.tracking._tracking_service.client: 🏃 View run Gradient Boosting Classifier at: http://127.0.0.1:5000/#/experiments/494945949872346356/runs/bbb5524630e846c7b78ae2917f2c2733.
2026/08/23 10:29:36 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/494945949872346356.


...

Fitting 10 folds for each of 25 candidates, totalling 250 fits


2026/08/23 10:54:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:54:40 INFO mlflow.tracking._tracking_service.client: 🏃 View run Decision Tree Classifier at: http://127.0.0.1:5000/#/experiments/668694462194489806/runs/8b066f557f7f4b21b0760bafa4710244.
2026/08/23 10:54:40 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/668694462194489806.


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6500,0.6346,0.5000,0.0357,0.0667,0.0210,0.0504
1,0.6375,0.8141,1.0000,0.0645,0.1212,0.0779,0.2013
2,0.6375,0.4827,0.3333,0.0357,0.0645,-0.0035,-0.0069
3,0.6125,0.6775,0.6667,0.0625,0.1143,0.0491,0.1074
4,0.5750,0.5281,0.3333,0.0303,0.0556,-0.0142,-0.0317
5,0.6375,0.8117,1.0000,0.0938,0.1714,0.1104,0.2417
6,0.6500,0.6667,0.3333,0.0370,0.0667,-0.0009,-0.0017
7,0.7125,0.7468,0.6667,0.0833,0.1481,0.0873,0.1579
8,0.7000,0.7294,0.6667,0.0800,0.1429,0.0813,0.1508


2026/08/23 10:54:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/23 10:54:43 INFO mlflow.tracking._tracking_service.client: 🏃 View run Decision Tree Classifier at: http://127.0.0.1:5000/#/experiments/668694462194489806/runs/39fd7384fa9a4dff9bdc1620d7546d2a.
2026/08/23 10:54:43 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/668694462194489806.


Transformation Pipeline and Model Successfully Saved


In [20]:
# Save metrics
metrics = {}
for col, model in best_models.items():
    print(f'{col} predictor: {type(model[0]).__name__} {model[2]} "Age" and "BMI" columns. (AUC: {model[1]})')

    metrics[col] = model[1]

with open(model_folder_path / "model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

Disease predictor: CalibratedClassifierCV with "Age" and "BMI" columns. (AUC: 0.5804)
Heart Disease predictor: CalibratedClassifierCV without "Age" and "BMI" columns. (AUC: 0.5774)
Diabetes predictor: CalibratedClassifierCV without "Age" and "BMI" columns. (AUC: 0.5707)
Stroke predictor: CalibratedClassifierCV without "Age" and "BMI" columns. (AUC: 0.5856)
Kidney Disease predictor: CalibratedClassifierCV without "Age" and "BMI" columns. (AUC: 0.5984)
Cancer predictor: CalibratedClassifierCV with "Age" and "BMI" columns. (AUC: 0.5626)
Alzheimer's Disease predictor: CalibratedClassifierCV with "Age" and "BMI" columns. (AUC: 0.5941)
COPD predictor: CalibratedClassifierCV with "Age" and "BMI" columns. (AUC: 0.6057)
Liver Disease predictor: CalibratedClassifierCV with "Age" and "BMI" columns. (AUC: 0.5415)
Parkinson's Disease predictor: CalibratedClassifierCV without "Age" and "BMI" columns. (AUC: 0.5415)
Tuberculosis predictor: CalibratedClassifierCV with "Age" and "BMI" columns. (AUC: 0.6